# Análisis exploratorio de **Viajes de Taxi** en **Chicago**

Este notebook forma parte de **Patrones Lab**, una serie de proyectos pensados para explorar datos reales con un enfoque claro, visual y reproducible. En este caso, el trabajo se centra en una base de **viajes de taxi en Chicago**, con el objetivo de ordenar los datos, revisar su calidad, construir variables útiles y empezar a detectar patrones en duración, distancia, tarifa y zonas de la ciudad.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

Apago los warnings para que la lectura del notebook quede más limpia.

In [2]:
import warnings
warnings.filterwarnings("ignore")

### Carpeta del proyecto
Detecto desde dónde corre el notebook y fijo la raíz del proyecto.

In [3]:
cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    PROJECT_DIR = cwd.parent
else:
    PROJECT_DIR = cwd

print("Project dir:", PROJECT_DIR.name)
print("Working dir:", cwd.name)

Project dir: 2026-03_taxi-trip-chicago
Working dir: notebooks


Armo rutas de trabajo

In [4]:
DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_STAGING = PROJECT_DIR / "data" / "staging"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"

TRIPS_FILE = DATA_RAW / "Taxi_Trips_20260324.csv"

### Cargo la base
Leo el CSV y miro tamaño más primeras filas para ver con qué arranco.

In [5]:
df = pd.read_csv(TRIPS_FILE)
print("trips shape:", df.shape)
df.head()

trips shape: (14219363, 23)


,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,...,Extras,Trip Total,Payment Type,Company,Pickup Centroid Latitude,Pickup Centroid Longitude,Pickup Centroid Location,Dropoff Centroid Latitude,Dropoff Centroid Longitude,Dropoff Centroid Location
0,0000184e7cd53cee95af32eba49c44e4d20adcd8,f538e6b729d1aaad4230e9dcd9dc2fd9a168826ddadbd6...,01/19/2024 05:00:00 PM,01/19/2024 06:00:00 PM,4051.0,17.12,1.703198e+10,1.703132e+10,76.0,32.0,...,4.0,60.00,Credit Card,Flash Cab,41.979071,-87.903040,POINT (-87.9030396611 41.9790708201),41.884987,-87.620993,POINT (-87.6209929134 41.8849871918)
1,000072ee076c9038868e239ca54185eb43959db0,e51e2c30caec952b40b8329a68b498e18ce8a1f40fa75c...,01/28/2024 02:30:00 PM,01/28/2024 03:00:00 PM,1749.0,12.70,NaN,NaN,6.0,NaN,...,0.0,33.75,Cash,Flash Cab,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),NaN,NaN,NaN
2,000074019d598c2b1d6e77fbae79e40b0461a2fc,aeb280ef3be3e27e081eb6e76027615b0d40925b84d3eb...,01/05/2024 09:00:00 AM,01/05/2024 09:00:00 AM,517.0,3.39,NaN,NaN,6.0,8.0,...,1.0,14.69,Mobile,Taxicab Insurance Agency Llc,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),41.899602,-87.633308,POINT (-87.6333080367 41.899602111)
3,00007572c5f92e2ff067e6f838a5ad74e83665d3,7d21c2ca227db8f27dda96612bfe5520ab408fa9a462c8...,01/22/2024 08:45:00 AM,01/22/2024 09:30:00 AM,2050.0,15.06,NaN,NaN,76.0,NaN,...,5.5,56.56,Credit Card,Globe Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),NaN,NaN,NaN
4,00007c3e7546e2c7d15168586943a9c22c3856cf,8ef1056519939d511d24008e394f83e925d2539d668a00...,01/18/2024 07:15:00 PM,01/18/2024 07:30:00 PM,1004.0,1.18,1.703184e+10,1.703184e+10,32.0,32.0,...,0.0,19.66,Mobile,5 Star Taxi,41.880994,-87.632746,POINT (-87.6327464887 41.8809944707),41.880994,-87.632746,POINT (-87.6327464887 41.8809944707)


### Primer vistazo a la estructura
Reviso tipos de datos, nulos y forma general de la base.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14219363 entries, 0 to 14219362
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     object 
 1   Taxi ID                     object 
 2   Trip Start Timestamp        object 
 3   Trip End Timestamp          object 
 4   Trip Seconds                float64
 5   Trip Miles                  float64
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Fare                        float64
 11  Tips                        float64
 12  Tolls                       float64
 13  Extras                      float64
 14  Trip Total                  float64
 15  Payment Type                object 
 16  Company                     object 
 17  Pickup Centroid Latitude    float64
 18  Pickup Centroid Longitude   float64
 19  Pickup Centroid Loc

Ordeno los nombres para que queden fáciles de usar.

In [7]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
)

In [8]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'pickup_centroid_location',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
       'dropoff_centroid__location'],
      dtype='object')

Corrijo una columna que vino con un guion bajo de más.

In [9]:
df = df.rename(columns={
    "dropoff_centroid__location": "dropoff_centroid_location"
})

### Fechas al formato correcto
Convierto inicio y fin del viaje a fecha real para poder trabajarlos bien.

In [10]:
df["trip_start_timestamp"] = pd.to_datetime(
    df["trip_start_timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

df["trip_end_timestamp"] = pd.to_datetime(
    df["trip_end_timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

In [11]:
df[["trip_start_timestamp", "trip_end_timestamp"]].head()

,trip_start_timestamp,trip_end_timestamp
0,2024-01-19 17:00:00,2024-01-19 18:00:00
1,2024-01-28 14:30:00,2024-01-28 15:00:00
2,2024-01-05 09:00:00,2024-01-05 09:00:00
3,2024-01-22 08:45:00,2024-01-22 09:30:00
4,2024-01-18 19:15:00,2024-01-18 19:30:00


Aseguro que montos, distancias y coordenadas queden en formato numérico.

In [12]:
cols_num = [
    "trip_seconds",
    "trip_miles",
    "fare",
    "tips",
    "tolls",
    "extras",
    "trip_total",
    "pickup_centroid_latitude",
    "pickup_centroid_longitude",
    "dropoff_centroid_latitude",
    "dropoff_centroid_longitude"
]

for c in cols_num:
    df[c] = pd.to_numeric(df[c], errors="coerce")

Hago lo mismo con tractos y community areas para dejarlos consistentes.

In [13]:
cols_id_geo = [
    "pickup_census_tract",
    "dropoff_census_tract",
    "pickup_community_area",
    "dropoff_community_area"
]

for c in cols_id_geo:
    df[c] = pd.to_numeric(df[c], errors="coerce")

Vuelvo a mirar la estructura para validar los cambios hechos hasta acá.

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14219363 entries, 0 to 14219362
Data columns (total 23 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   trip_id                     object        
 1   taxi_id                     object        
 2   trip_start_timestamp        datetime64[ns]
 3   trip_end_timestamp          datetime64[ns]
 4   trip_seconds                float64       
 5   trip_miles                  float64       
 6   pickup_census_tract         float64       
 7   dropoff_census_tract        float64       
 8   pickup_community_area       float64       
 9   dropoff_community_area      float64       
 10  fare                        float64       
 11  tips                        float64       
 12  tolls                       float64       
 13  extras                      float64       
 14  trip_total                  float64       
 15  payment_type                object        
 16  company         

Reviso unas filas para ver cómo quedó el dataset después de la limpieza inicial.

In [15]:
df.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location
0,0000184e7cd53cee95af32eba49c44e4d20adcd8,f538e6b729d1aaad4230e9dcd9dc2fd9a168826ddadbd6...,2024-01-19 17:00:00,2024-01-19 18:00:00,4051.0,17.12,1.703198e+10,1.703132e+10,76.0,32.0,...,4.0,60.00,Credit Card,Flash Cab,41.979071,-87.903040,POINT (-87.9030396611 41.9790708201),41.884987,-87.620993,POINT (-87.6209929134 41.8849871918)
1,000072ee076c9038868e239ca54185eb43959db0,e51e2c30caec952b40b8329a68b498e18ce8a1f40fa75c...,2024-01-28 14:30:00,2024-01-28 15:00:00,1749.0,12.70,NaN,NaN,6.0,NaN,...,0.0,33.75,Cash,Flash Cab,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),NaN,NaN,NaN
2,000074019d598c2b1d6e77fbae79e40b0461a2fc,aeb280ef3be3e27e081eb6e76027615b0d40925b84d3eb...,2024-01-05 09:00:00,2024-01-05 09:00:00,517.0,3.39,NaN,NaN,6.0,8.0,...,1.0,14.69,Mobile,Taxicab Insurance Agency Llc,41.944227,-87.655998,POINT (-87.6559981815 41.9442266014),41.899602,-87.633308,POINT (-87.6333080367 41.899602111)
3,00007572c5f92e2ff067e6f838a5ad74e83665d3,7d21c2ca227db8f27dda96612bfe5520ab408fa9a462c8...,2024-01-22 08:45:00,2024-01-22 09:30:00,2050.0,15.06,NaN,NaN,76.0,NaN,...,5.5,56.56,Credit Card,Globe Taxi,41.980264,-87.913625,POINT (-87.913624596 41.9802643146),NaN,NaN,NaN
4,00007c3e7546e2c7d15168586943a9c22c3856cf,8ef1056519939d511d24008e394f83e925d2539d668a00...,2024-01-18 19:15:00,2024-01-18 19:30:00,1004.0,1.18,1.703184e+10,1.703184e+10,32.0,32.0,...,0.0,19.66,Mobile,5 Star Taxi,41.880994,-87.632746,POINT (-87.6327464887 41.8809944707),41.880994,-87.632746,POINT (-87.6327464887 41.8809944707)


### Aplico filtros duros
Me quedo solo con viajes con duración, distancia y tarifa mínimas razonables.

In [16]:
# FILTROS DUROS

df = df[
    (df["trip_seconds"] >= 50) &
    (df["trip_miles"] >= 0.25) &
    (df["fare"] >= 3.25)
].copy()

Veo en qué columnas faltan más datos.

In [17]:
df.isna().sum().sort_values(ascending=False)

dropoff_census_tract          7266113
pickup_census_tract           7084540
dropoff_community_area        1100012
dropoff_centroid_longitude    1032673
dropoff_centroid_location     1032673
dropoff_centroid_latitude     1032673
pickup_community_area          234374
pickup_centroid_location       232969
pickup_centroid_latitude       232969
pickup_centroid_longitude      232969
taxi_id                             7
trip_id                             0
trip_seconds                        0
trip_end_timestamp                  0
trip_start_timestamp                0
trip_miles                          0
trip_total                          0
tolls                               0
tips                                0
fare                                0
extras                              0
payment_type                        0
company                             0
dtype: int64

Paso el mismo chequeo a porcentaje para compararlo mejor.

In [18]:
(df.isna().mean() * 100).sort_values(ascending=False).round(2)

dropoff_census_tract          57.38
pickup_census_tract           55.95
dropoff_community_area         8.69
dropoff_centroid_longitude     8.16
dropoff_centroid_location      8.16
dropoff_centroid_latitude      8.16
pickup_community_area          1.85
pickup_centroid_location       1.84
pickup_centroid_latitude       1.84
pickup_centroid_longitude      1.84
taxi_id                        0.00
trip_id                        0.00
trip_seconds                   0.00
trip_end_timestamp             0.00
trip_start_timestamp           0.00
trip_miles                     0.00
trip_total                     0.00
tolls                          0.00
tips                           0.00
fare                           0.00
extras                         0.00
payment_type                   0.00
company                        0.00
dtype: float64

### Creo variables de fecha
Saco año, mes, día, hora y fin de semana a partir del inicio del viaje.

In [19]:
df["trip_date"] = df["trip_start_timestamp"].dt.date
df["trip_year"] = df["trip_start_timestamp"].dt.year
df["trip_month"] = df["trip_start_timestamp"].dt.month
df["trip_day"] = df["trip_start_timestamp"].dt.day
df["trip_hour"] = df["trip_start_timestamp"].dt.hour
df["trip_weekday"] = df["trip_start_timestamp"].dt.day_name()
df["trip_weekday_num"] = df["trip_start_timestamp"].dt.weekday

df["is_weekend"] = df["trip_weekday_num"].isin([5, 6])

In [20]:
df[["trip_start_timestamp", "trip_date", "trip_year", "trip_month", "trip_hour", "trip_weekday", "trip_weekday_num"]].head()

,trip_start_timestamp,trip_date,trip_year,trip_month,trip_hour,trip_weekday,trip_weekday_num
0,2024-01-19 17:00:00,2024-01-19,2024,1,17,Friday,4
1,2024-01-28 14:30:00,2024-01-28,2024,1,14,Sunday,6
2,2024-01-05 09:00:00,2024-01-05,2024,1,9,Friday,4
3,2024-01-22 08:45:00,2024-01-22,2024,1,8,Monday,0
4,2024-01-18 19:15:00,2024-01-18,2024,1,19,Thursday,3


Miro estadísticos simples de tiempo, distancia y montos.

In [21]:
df[["trip_seconds", "trip_miles", "fare", "tips", "tolls", "extras", "trip_total"]].describe()

,trip_seconds,trip_miles,fare,tips,tolls,extras,trip_total
count,1.266262e+07,1.266262e+07,1.266262e+07,1.266262e+07,1.266262e+07,1.266262e+07,1.266262e+07
mean,1.312205e+03,7.341244e+00,2.256715e+01,2.818682e+00,2.942088e-02,1.986892e+00,2.765639e+01
std,1.526550e+03,7.600226e+00,1.860421e+01,4.175256e+00,2.175365e+00,7.219397e+00,2.465846e+01
min,5.000000e+01,2.500000e-01,3.250000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.250000e+00
25%,5.460000e+02,1.450000e+00,8.750000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.073000e+01
50%,9.910000e+02,4.200000e+00,1.600000e+01,0.000000e+00,0.000000e+00,0.000000e+00,1.850000e+01
75%,1.740000e+03,1.243000e+01,3.416000e+01,4.000000e+00,0.000000e+00,2.000000e+00,4.200000e+01
max,8.639600e+04,3.397800e+03,9.999750e+03,4.000000e+02,4.444440e+03,5.559500e+03,9.999750e+03


Chequeo desde qué fecha hasta qué fecha llegan los viajes.

In [22]:
df["trip_start_timestamp"].min(), df["trip_start_timestamp"].max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2026-03-01 00:00:00'))

Marco si cada viaje tiene monto y algo de información geográfica en origen y destino.

In [23]:
df["flag_monto"] = df["fare"].notna()

df["flag_pickup_geo"] = (
    df["pickup_community_area"].notna() |
    df["pickup_centroid_latitude"].notna() |
    df["pickup_centroid_longitude"].notna()
)

df["flag_dropoff_geo"] = (
    df["dropoff_community_area"].notna() |
    df["dropoff_centroid_latitude"].notna() |
    df["dropoff_centroid_longitude"].notna()
)

In [24]:
df[["flag_monto", "flag_pickup_geo", "flag_dropoff_geo"]].head()

,flag_monto,flag_pickup_geo,flag_dropoff_geo
0,True,True,True
1,True,True,False
2,True,True,True
3,True,True,False
4,True,True,True


### Paso duración a minutos y horas
Creo versiones más cómodas de la duración del viaje.

In [25]:
df["trip_minutes"] = df["trip_seconds"] / 60
df["trip_hours"] = df["trip_seconds"] / 3600

### Recalcular el total por partes
Sumo tarifa, propina, peajes y extras para tener un total armado por separado.

In [26]:
df["total_calculado"] = df["fare"] + df["tips"] + df["tolls"] + df["extras"]

In [27]:
#   df["diff_total"] = df["trip_total"] - df["total_calculado"]

In [28]:
# df["flag_total_consistente"] = df["diff_total"].abs() <= 0.01

In [29]:
# df["flag_total_consistente"].mean()

### Calculo velocidad estimada
Armo una velocidad simple usando distancia y duración.

In [30]:
df["speed_mph"] = df["trip_miles"] / df["trip_hours"]

### Calculo tarifa por milla
Llevo la tarifa a una métrica más comparable entre viajes.

In [31]:
df["fare_per_mile"] = df["fare"] / df["trip_miles"]

### Calculo total por milla
Hago lo mismo, pero usando el total completo del viaje.

In [32]:
df["trip_total_per_mile"] = df["trip_total"] / df["trip_miles"]

### Calculo tarifa por minuto
Paso la tarifa a una lógica por tiempo.

In [33]:
df["fare_per_minute"] = df["fare"] / df["trip_minutes"]

### Calculo total por minuto
Completo la familia de ratios con el total por minuto.

In [34]:
df["trip_total_per_minute"] = df["trip_total"] / df["trip_minutes"]

### Limpio infinitos en ratios
Cambio valores infinitos por nulos para no arrastrar errores raros.

In [35]:
import numpy as np

cols_ratio = [
    "speed_mph",
    "fare_per_mile",
    "trip_total_per_mile",
    "fare_per_minute",
    "trip_total_per_minute"
]

for c in cols_ratio:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan)

### Calculo porcentaje de propina
Mido qué parte de la tarifa representa la propina.

In [36]:
df["tip_pct"] = (df["tips"] / df["fare"]) * 100
df["tip_pct"] = df["tip_pct"].replace([np.inf, -np.inf], np.nan)

Señalo los registros que tienen las piezas básicas para análisis operativo.

In [37]:
df["flag_trip_core"] = (
    df["trip_seconds"].notna() &
    df["trip_miles"].notna() &
    df["trip_start_timestamp"].notna()
)

### Agrupo por franja horaria
Paso la hora a una banda simple para leer mejor los viajes a lo largo del día.

In [38]:
df["time_band"] = "night"

df.loc[df["trip_hour"].between(6, 11), "time_band"] = "morning"
df.loc[df["trip_hour"].between(12, 17), "time_band"] = "afternoon"
df.loc[df["trip_hour"].between(18, 23), "time_band"] = "evening"

In [39]:
df[
    [
        "trip_year", "trip_month", "trip_hour", "is_weekend", "time_band",
        "trip_minutes", "trip_hours", "total_calculado", "speed_mph", "fare_per_mile",
        "trip_total_per_mile", "fare_per_minute", "trip_total_per_minute",
        "tip_pct", "flag_trip_core"
    ]
].head()

,trip_year,trip_month,trip_hour,is_weekend,time_band,trip_minutes,trip_hours,total_calculado,speed_mph,fare_per_mile,trip_total_per_mile,fare_per_minute,trip_total_per_minute,tip_pct,flag_trip_core
0,2024,1,17,False,afternoon,67.516667,1.125278,59.50,15.214021,2.657710,3.504673,0.673908,0.888669,21.978022,True
1,2024,1,14,True,afternoon,29.150000,0.485833,33.75,26.140652,2.657480,2.657480,1.157804,1.157804,0.000000,True
2,2024,1,9,False,morning,8.616667,0.143611,14.69,23.605416,3.218289,4.333333,1.266151,1.704836,25.481210,True
3,2024,1,8,False,morning,34.166667,0.569444,56.06,26.446829,2.606242,3.755644,1.148780,1.655415,28.815287,True
4,2024,1,19,False,evening,16.733333,0.278889,19.66,4.231076,13.508475,16.661017,0.952590,1.174900,23.337516,True


In [40]:
df[
    [
        "trip_minutes", "trip_hours", "speed_mph",
        "fare_per_mile", "trip_total_per_mile", "fare_per_minute",
        "trip_total_per_minute", "tip_pct"
    ]
].describe()

,trip_minutes,trip_hours,speed_mph,fare_per_mile,trip_total_per_mile,fare_per_minute,trip_total_per_minute,tip_pct
count,1.266262e+07,1.266262e+07,1.266262e+07,1.266262e+07,1.266262e+07,1.266262e+07,1.266262e+07,1.266262e+07
mean,2.187008e+01,3.645013e-01,1.843971e+01,5.019315e+00,6.118079e+00,1.144852e+00,1.402138e+00,1.313798e+01
std,2.544251e+01,4.240418e-01,5.355726e+01,6.576044e+00,1.230238e+01,1.866986e+00,2.077519e+00,1.806629e+01
min,8.333333e-01,1.388889e-02,1.315751e-02,1.240840e-03,2.127154e-03,2.518827e-03,2.938116e-03,0.000000e+00
25%,9.100000e+00,1.516667e-01,9.473684e+00,2.679404e+00,3.289329e+00,8.169692e-01,9.444444e-01,0.000000e+00
50%,1.651667e+01,2.752778e-01,1.487386e+01,3.617711e+00,4.362416e+00,1.018006e+00,1.210669e+00,0.000000e+00
75%,2.900000e+01,4.833333e-01,2.507104e+01,5.818966e+00,6.932773e+00,1.314103e+00,1.616162e+00,2.303740e+01
max,1.439933e+03,2.399889e+01,1.410335e+05,1.105789e+04,2.228032e+04,3.930536e+03,3.930536e+03,2.613333e+03


In [41]:
df.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'pickup_centroid_location',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
       'dropoff_centroid_location', 'trip_date', 'trip_year', 'trip_month',
       'trip_day', 'trip_hour', 'trip_weekday', 'trip_weekday_num',
       'is_weekend', 'flag_monto', 'flag_pickup_geo', 'flag_dropoff_geo',
       'trip_minutes', 'trip_hours', 'total_calculado', 'speed_mph',
       'fare_per_mile', 'trip_total_per_mile', 'fare_per_minute',
       'trip_total_per_minute', 'tip_pct', 'flag_trip_core', 'time_band'],
      dtype='object')

### Ajusto casos imposibles
Pongo en nulo los ratios que no tienen sentido por divisiones o valores no válidos.

In [42]:
df.loc[df["trip_hours"] <= 0, "speed_mph"] = np.nan

df.loc[df["trip_miles"] <= 0, "fare_per_mile"] = np.nan
df.loc[df["trip_miles"] <= 0, "trip_total_per_mile"] = np.nan

df.loc[df["trip_minutes"] <= 0, "fare_per_minute"] = np.nan
df.loc[df["trip_minutes"] <= 0, "trip_total_per_minute"] = np.nan

df.loc[df["fare"] <= 0, "tip_pct"] = np.nan

Busco duplicados por trip_id

In [43]:
df["trip_id"].duplicated().sum()

np.int64(0)

Busco duplicados por combinación clave

In [44]:
df.duplicated(subset=["taxi_id", "trip_start_timestamp", "trip_end_timestamp"]).sum()

np.int64(38738)

Dejo juntas las variables numéricas que quiero mirar con más detalle.

In [45]:
cols = [
    "trip_seconds",
    "trip_miles",
    "fare",
    "tips",
    "tolls",
    "extras",
    "trip_total",
    "trip_minutes",
    "trip_hours",
    "speed_mph",
    "fare_per_mile",
    "trip_total_per_mile",
    "fare_per_minute",
    "trip_total_per_minute",
    "tip_pct"
]

#   df[cols].agg(["min", "max"]).T

Reviso mínimos, medianas y valores altos para detectar casos raros.

In [46]:
df[cols].describe(percentiles=[0.01, 0.02, 0.05, 0.50, 0.95, 0.98, 0.99]).round(2).T

,count,mean,std,min,1%,2%,5%,50%,95%,98%,99%,max
trip_seconds,12662620.0,1312.20,1526.55,50.00,162.00,193.00,266.00,991.00,3258.00,4020.00,4620.00,86396.00
trip_miles,12662620.0,7.34,7.60,0.25,0.38,0.46,0.61,4.20,18.53,23.07,27.30,3397.80
fare,12662620.0,22.57,18.60,3.25,4.50,4.75,5.50,16.00,50.37,64.00,72.00,9999.75
tips,12662620.0,2.82,4.18,0.00,0.00,0.00,0.00,0.00,10.94,13.65,15.70,400.00
tolls,12662620.0,0.03,2.18,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,4444.44
extras,12662620.0,1.99,7.22,0.00,0.00,0.00,0.00,0.00,6.00,23.00,30.00,5559.50
trip_total,12662620.0,27.66,24.66,3.25,4.75,5.25,6.25,18.50,66.50,83.75,97.89,9999.75
trip_minutes,12662620.0,21.87,25.44,0.83,2.70,3.22,4.43,16.52,54.30,67.00,77.00,1439.93
trip_hours,12662620.0,0.36,0.42,0.01,0.04,0.05,0.07,0.28,0.90,1.12,1.28,24.00
speed_mph,12662620.0,18.44,53.56,0.01,2.85,4.33,5.89,14.87,41.33,47.28,50.86,141033.50


Inspecciono los casos más largos en distancia para ver si suenan razonables.

In [47]:
df.sort_values("trip_miles", ascending=False)[
    ["trip_id", "trip_start_timestamp", "trip_seconds", "trip_miles", "fare", "trip_total", "payment_type", "company"]
].head(20)

,trip_id,trip_start_timestamp,trip_seconds,trip_miles,fare,trip_total,payment_type,company
1727058,8bb9ae67ccafb402045fd52e5f2ba31ef2963835,2024-04-21 07:15:00,8092.0,3397.80,67.00,67.00,Cash,Chicago Independents
673741,8656832cf4c4d23c8bbb728f43bd645020b1258f,2024-02-10 10:00:00,27853.0,3093.47,9999.75,9999.75,Cash,Blue Ribbon Taxi Association
2853227,80e84d4e2ee8e03cf26c994714700b6a7eacf4c6,2024-06-25 14:00:00,2313.0,3017.61,44.25,58.50,Credit Card,Sun Taxi
6373816,c0198d5377db37f1508aa57c8a299d18f2c5b566,2024-12-01 09:30:00,14925.0,2949.41,9999.75,9999.75,Cash,Blue Ribbon Taxi Association
2119521,522fc9a30df54459fbee9558749bbd29674ae700,2024-05-28 16:00:00,72.0,2820.67,3.50,6.00,Cash,Flash Cab
5612768,45c7b37478c4cd41b3abe89bbcf390a8eb2fc9ab,2024-11-25 13:45:00,1732.0,2537.51,42.75,56.70,Credit Card,Choice Taxi Association Inc
3616284,65d80b26805e9548766a10a894848f882374660b,2024-08-06 08:30:00,33950.0,2265.43,9999.75,9999.75,Cash,Blue Ribbon Taxi Association
6151458,52c4902403064408e66026dfa323bb13925b5eb7,2024-12-24 00:15:00,1585.0,2166.39,30.00,30.00,Prcard,5 Star Taxi
188038,6e4aecb7dab42a2109ec97a76e408699e2ad0c62,2024-01-31 12:45:00,56.0,1626.85,3668.50,3668.50,Cash,Blue Ribbon Taxi Association
5742445,8139408b3729602119b3dafc205a7f6debfc28ef,2024-11-06 10:00:00,32809.0,1127.86,5409.25,5409.25,Cash,Blue Ribbon Taxi Association


Hago la misma revisión, pero con las duraciones más grandes.

In [48]:
df.sort_values("trip_seconds", ascending=False)[
    ["trip_id", "trip_start_timestamp", "trip_seconds", "trip_miles", "fare", "trip_total", "payment_type", "company"]
].head(20)

,trip_id,trip_start_timestamp,trip_seconds,trip_miles,fare,trip_total,payment_type,company
8123455,5224d3ab360fc1ba9a660f92b7c471bc0383818d,2025-03-19 18:15:00,86396.0,1.47,9.25,10.25,Cash,Taxicab Insurance Agency Llc
1495930,1e10bbfb70b9ac63d8c37f191b8760eae77b4399,2024-04-24 22:15:00,86379.0,15.34,38.75,42.75,Cash,Chicago Independents
5704468,6fc043c9bf81448058a3df76fb89837db9f6ac31,2024-11-13 21:00:00,86322.0,18.51,47.75,55.75,Cash,Flash Cab
3030978,c8c44e4d09621abefef0d5fbd9ee3d13c7d6515d,2024-06-20 21:00:00,86265.0,10.48,32.25,32.25,Cash,Globe Taxi
9201801,190b6ad45a46d90fd547bd6a96f40c394a911921,2025-06-13 20:45:00,86144.0,241.03,902.25,902.25,Cash,Blue Ribbon Taxi Association
55591,20965e71f44cb0541beea63f5bea9f02357eed90,2024-01-10 15:30:00,86135.0,2.42,38.75,38.75,Cash,Flash Cab
5526904,1e5319a4c87c02b57e883576b05b1de8371e0862,2024-11-03 05:30:00,86078.0,10.46,26.75,26.75,Cash,5 Star Taxi
6811507,b43fa82b87a331338bf76ab381679c37fa2dd82f,2025-01-30 17:45:00,86067.0,4.95,17.00,17.00,Cash,Blue Ribbon Taxi Association
9636045,c49564de3b8debee7242f768bbcbaf6bd68ddbf3,2025-06-15 22:00:00,86036.0,200.10,811.25,811.25,Cash,Blue Ribbon Taxi Association
6466711,edd7f0d9dcfe36d77bb8bf97c58f538e3192a1d7,2024-12-01 13:15:00,85952.0,11.18,32.75,32.75,Cash,Flash Cab


### Exporto staging en parquet

In [49]:
#   df.to_parquet(DATA_STAGING / "taxi_trips_chicago_staging.parquet", index=False)

### Exportar recorte 2026

In [50]:
#   # ME QUEDO CON EL 2026 PARA TRAGBAJAR EN DASHBOARDS
#   df_2026 = df[df["trip_year"] == 2026].copy()
#   
#   df_2026.to_csv(DATA_STAGING / "taxi_trips_chicago_2026.csv", index=False)